# RoadLab: a reproducible severity experiment

AI-assisted study implementation. Read the model card before interpreting results. Run from the repository root in the pinned Python 3.11 environment. This notebook is unexecuted; delivered metrics come from the script run.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from roadlab.core import ROOT


## 1. Rebuild the experiment
This calls the same implementation used by CI. Review `roadlab/train.py`: dataset checksum, grouped splitting, training-only transforms, validation selection and locked test evaluation.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "roadlab.train"], check=True)


## 2. Compare candidates
Why is accuracy alone misleading? Which class is rarest? Explain why a validation winner is kept even if another model has a better test metric.

In [ ]:
report = json.loads((ROOT / "artifacts/metrics.json").read_text())
pd.DataFrame({name: {"validation_macro_f1": result["validation_macro_f1"], **{k: result["test"][k] for k in ["accuracy", "balanced_accuracy", "macro_f1"]}} for name, result in report["models"].items()}).T


In [ ]:
selected = report["models"][report["selected"]]["test"]
pd.DataFrame(selected["per_class"]).T


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "artifacts/evaluation.png")))


## 3. Inspect the split and interpretation
There is no geographic or temporal holdout. Identical feature profiles are grouped, but real incident IDs are absent. The bootstrap interval conditions on a fixed trained model and test set; it is not a deployment guarantee.

In [ ]:
report["split"], report["selected_macro_f1_bootstrap_95_ci"]


## 4. Exercise the actual API locally
This uses the same route called by the interface, not a second prediction implementation. Scores are uncalibrated. Change one input and explain what you can and cannot conclude.

In [ ]:
from fastapi.testclient import TestClient
from roadlab.api import app
client = TestClient(app)
example = client.get("/api/metadata").json()["examples"][0]
client.post("/api/predict", json={"records": [example]}).json()


## 5. Your own reflection
Write your own explanation of leakage, macro-F1, class imbalance, grouping, API validation and limitations. Disclose AI assistance and do not present this notebook as unaided work.